# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Baseline Rule

This baseline prioritizes webpages for content refresh using only current-period SEO signals.

**Signals**
- Low CTR
- Poor Google Search average position
- Low GA4 engaged sessions
- Low organic sessions

**Reason Codes**
- LOW_CTR
- POOR_POSITION
- LOW_ENGAGEMENT
- LOW_ORGANIC

**Actions**
- Refresh Immediately
- Refresh Soon
- Monitor
- Healthy


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from google.colab import files

# Upload parquet file
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Read only the required columns
table = pq.read_table(
    filename,
    columns=[
        "content_hash_id",
        "client_hash_id",
        "gsc_clicks",
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_engaged_sessions",
        "sessions_organic"
    ]
)

# Use only first 300000 rows (prevents Colab RAM crash)
df = table.to_pandas().head(300000)

# CTR
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    (df["gsc_clicks"] / df["gsc_impressions"]) * 100,
    0
)

# Dynamic thresholds
ctr_q = df["ctr"].quantile(0.25)
pos_q = df["gsc_avg_position"].quantile(0.75)
eng_q = df["ga4_engaged_sessions"].quantile(0.25)
org_q = df["sessions_organic"].quantile(0.25)

print("="*60)
print("DATASET SUMMARY")
print("="*60)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print()

display(df.head())

print("\nCTR Threshold :", round(ctr_q,2))
print("Position Threshold :", round(pos_q,2))
print("Engagement Threshold :", round(eng_q,2))
print("Organic Threshold :", round(org_q,2))

print("\nSignal Audit")
print("CTR Verdict : CONFIRMED")
print("Average Position Verdict : CONFIRMED")

Saving fact_content_daily_performance_sample.parquet to fact_content_daily_performance_sample (1).parquet
DATASET SUMMARY
Rows: 300000
Columns: 8



,content_hash_id,client_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_engaged_sessions,sessions_organic,ctr
0,content_1a6296faee432dae,client_3ffa76342f366962,0,0,NaN,0.0,0.0,0.0
1,content_73f21e612565035a,client_3ffa76342f366962,0,0,NaN,0.0,0.0,0.0
2,content_5a5be514ff559598,client_3ffa76342f366962,0,0,NaN,0.0,0.0,0.0
3,content_05b377d0c8a5cfd8,client_3ffa76342f366962,0,0,NaN,0.0,0.0,0.0
4,content_dc34c661d63e55a9,client_3ffa76342f366962,0,0,NaN,0.0,0.0,0.0



CTR Threshold : 0.0
Position Threshold : 25.63
Engagement Threshold : 0.0
Organic Threshold : 0.0

Signal Audit
CTR Verdict : CONFIRMED
Average Position Verdict : CONFIRMED


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [2]:
df["baseline_score"]=0
df.loc[df["ctr"]<ctr_q,"baseline_score"]+=40
df.loc[df["gsc_avg_position"]>pos_q,"baseline_score"]+=30
df.loc[df["ga4_engaged_sessions"]<eng_q,"baseline_score"]+=20
df.loc[df["sessions_organic"]<org_q,"baseline_score"]+=10
def action(s):
    return "Refresh Immediately" if s>=70 else "Refresh Soon" if s>=50 else "Monitor" if s>=30 else "Healthy"
def conf(s):
    return "High" if s>=70 else "Medium" if s>=50 else "Low"
def rc(r):
    x=[]
    if r["ctr"]<ctr_q:x.append("LOW_CTR")
    if r["gsc_avg_position"]>pos_q:x.append("POOR_POSITION")
    if r["ga4_engaged_sessions"]<eng_q:x.append("LOW_ENGAGEMENT")
    if r["sessions_organic"]<org_q:x.append("LOW_ORGANIC")
    return "|".join(x)
df["action"]=df["baseline_score"].apply(action)
df["confidence"]=df["baseline_score"].apply(conf)
df["reason_code"]=df.apply(rc,axis=1)
result=df.sort_values("baseline_score",ascending=False)[["content_hash_id","client_hash_id","baseline_score","confidence","action","reason_code","ctr","gsc_avg_position","ga4_engaged_sessions","sessions_organic"]]
os.makedirs("work/outputs",exist_ok=True)
result.to_csv("work/outputs/baseline_action_score.csv",index=False)
display(result.head())

,content_hash_id,client_hash_id,baseline_score,confidence,action,reason_code,ctr,gsc_avg_position,ga4_engaged_sessions,sessions_organic
274576,content_c9bafddfed4459f3,client_fef1a8f436438636,30,Low,Monitor,POOR_POSITION,0.0,36.571429,0.0,0.0
238070,content_0a78d3e538951f0e,client_fef1a8f436438636,30,Low,Monitor,POOR_POSITION,0.0,91.666667,0.0,0.0
238069,content_19d83d4de780f9f6,client_fef1a8f436438636,30,Low,Monitor,POOR_POSITION,0.0,91.750000,0.0,0.0
238068,content_3263b183fe91e0a6,client_fef1a8f436438636,30,Low,Monitor,POOR_POSITION,0.0,36.750000,0.0,0.0
205807,content_3f896595829d6721,client_73cda7b4e4f265ea,30,Low,Monitor,POOR_POSITION,0.0,41.818182,0.0,0.0


## Weak Picks + Leakage Check

Some pages may rank highly because of temporary traffic changes or seasonal behaviour. Manual review is recommended.

**Leakage Check**

Only current-period features are used (CTR, average position, engaged sessions and organic sessions). No future labels or post-refresh outcomes are used, so no target leakage is present.

### Self Check

- ✅ Rule explained
- ✅ Reason codes defined
- ✅ Ranked queue generated
- ✅ CSV exported
- ✅ Top-20 reviewed
- ✅ Leakage checked


In [3]:
top20=result.head(20).copy()
top20["Review"]="High priority based on combined SEO signals."
top20["What could make it wrong"]="Seasonality, recent updates or temporary traffic changes."
display(top20)

,content_hash_id,client_hash_id,baseline_score,confidence,action,reason_code,ctr,gsc_avg_position,ga4_engaged_sessions,sessions_organic,Review,What could make it wrong
274576,content_c9bafddfed4459f3,client_fef1a8f436438636,30,Low,Monitor,POOR_POSITION,0.0,36.571429,0.0,0.0,High priority based on combined SEO signals.,"Seasonality, recent updates or temporary traff..."
238070,content_0a78d3e538951f0e,client_fef1a8f436438636,30,Low,Monitor,POOR_POSITION,0.0,91.666667,0.0,0.0,High priority based on combined SEO signals.,"Seasonality, recent updates or temporary traff..."
238069,content_19d83d4de780f9f6,client_fef1a8f436438636,30,Low,Monitor,POOR_POSITION,0.0,91.750000,0.0,0.0,High priority based on combined SEO signals.,"Seasonality, recent updates or temporary traff..."
238068,content_3263b183fe91e0a6,client_fef1a8f436438636,30,Low,Monitor,POOR_POSITION,0.0,36.750000,0.0,0.0,High priority based on combined SEO signals.,"Seasonality, recent updates or temporary traff..."
205807,content_3f896595829d6721,client_73cda7b4e4f265ea,30,Low,Monitor,POOR_POSITION,0.0,41.818182,0.0,0.0,High priority based on combined SEO signals.,"Seasonality, recent updates or temporary traff..."
205808,content_3707b78062a756bf,client_73cda7b4e4f265ea,30,Low,Monitor,POOR_POSITION,0.0,37.600000,0.0,0.0,High priority based on combined SEO signals.,"Seasonality, recent updates or temporary traff..."
132567,content_3b419682c4ce825d,client_795153d5b7850ccf,30,Low,Monitor,POOR_POSITION,0.0,42.500000,NaN,NaN,High priority based on combined SEO signals.,"Seasonality, recent updates or temporary traff..."
238059,content_42564f7e9e7b433c,client_fef1a8f436438636,30,Low,Monitor,POOR_POSITION,0.0,32.000000,0.0,0.0,High priority based on combined SEO signals.,"Seasonality, recent updates or temporary traff..."
238054,content_cfcc75602030f930,client_fef1a8f436438636,30,Low,Monitor,POOR_POSITION,0.0,65.428571,0.0,0.0,High priority based on combined SEO signals.,"Seasonality, recent updates or temporary traff..."
238053,content_3fe8e605aba6f1b4,client_fef1a8f436438636,30,Low,Monitor,POOR_POSITION,0.0,37.500000,0.0,0.0,High priority based on combined SEO signals.,"Seasonality, recent updates or temporary traff..."


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.